In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/prasadmalaiiiitk/dataset/sem-dataset/stage1/README.dataset.txt
/kaggle/input/datasets/prasadmalaiiiitk/dataset/sem-dataset/stage1/README.roboflow.txt
/kaggle/input/datasets/prasadmalaiiiitk/dataset/sem-dataset/stage1/valid/defect/crack_image_50_png.rf.ce120a785448870198c7c4134b11b314.jpg
/kaggle/input/datasets/prasadmalaiiiitk/dataset/sem-dataset/stage1/valid/defect/short_image_40_png.rf.03a34c0824f7c32983f413a8d8b0c492.jpg
/kaggle/input/datasets/prasadmalaiiiitk/dataset/sem-dataset/stage1/valid/defect/open_image_17_png.rf.071c538dac6a0a85711bdbe008349f4d.jpg
/kaggle/input/datasets/prasadmalaiiiitk/dataset/sem-dataset/stage1/valid/defect/short_image_42_png.rf.153c81d432112d1a71c365fb8292ea95.jpg
/kaggle/input/datasets/prasadmalaiiiitk/dataset/sem-dataset/stage1/valid/defect/LER_image_53_png.rf.3d93880deb034d22a9a0fc9d01582c3b.jpg
/kaggle/input/datasets/prasadmalaiiiitk/dataset/sem-dataset/stage1/valid/defect/short_image_34_png.rf.c6308e159eec266bbee981fe562ac5c7.

In [2]:
import os
from PIL import Image
from tqdm import tqdm
from torchvision import transforms

# -----------------------
# PATHS (IMPORTANT)
# -----------------------
INPUT_DIR = "/kaggle/input/datasets/prasadmalaiiiitk/dataset/sem-dataset/stage2/train"
OUTPUT_DIR = "/kaggle/working/augmented_stage2/train"

AUGMENTATIONS_PER_IMAGE = 3
TARGET_SIZE = 224

# -----------------------
# TRANSFORMS
# -----------------------
resize_transform = transforms.Compose([
    transforms.Resize(256),          # keep aspect ratio
    transforms.CenterCrop(TARGET_SIZE)
])

augmentation_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(TARGET_SIZE),
    transforms.RandomRotation(8),
    transforms.RandomResizedCrop(TARGET_SIZE, scale=(0.95, 1.05)),
    transforms.ColorJitter(brightness=0.08, contrast=0.08),
    transforms.ToTensor(),
    transforms.ToPILImage()
])

# -----------------------
# CREATE OUTPUT STRUCTURE
# -----------------------
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Classes found:", os.listdir(INPUT_DIR))

for class_name in os.listdir(INPUT_DIR):

    class_input_path = os.path.join(INPUT_DIR, class_name)

    if not os.path.isdir(class_input_path):
        continue

    class_output_path = os.path.join(OUTPUT_DIR, class_name)
    os.makedirs(class_output_path, exist_ok=True)

    print(f"Processing class: {class_name}")

    for img_name in tqdm(os.listdir(class_input_path)):
        img_path = os.path.join(class_input_path, img_name)

        try:
            image = Image.open(img_path).convert("RGB")
        except:
            continue

        # Save resized original
        resized = resize_transform(image)
        resized.save(os.path.join(class_output_path, img_name))

        # Generate augmented copies
        for i in range(AUGMENTATIONS_PER_IMAGE):
            augmented = augmentation_transform(image)
            new_name = f"{os.path.splitext(img_name)[0]}_aug_{i}.jpg"
            augmented.save(os.path.join(class_output_path, new_name))

print("✅ Augmentation Complete")

Classes found: ['LER', 'malformed_via', 'short', 'open', 'crack']
Processing class: LER


100%|██████████| 49/49 [00:01<00:00, 39.64it/s]


Processing class: malformed_via


100%|██████████| 49/49 [00:01<00:00, 43.32it/s]


Processing class: short


...

Processing class: open


100%|██████████| 57/57 [00:01<00:00, 31.54it/s]


Processing class: crack


100%|██████████| 51/51 [00:01<00:00, 46.30it/s]

✅ Augmentation Complete


In [3]:
import os
print(os.getcwd())

/kaggle/working


In [4]:
import shutil

shutil.make_archive(
    '/kaggle/working/augmented_stage2',
    'zip',
    '/kaggle/working/augmented_stage2'
)

'/kaggle/working/augmented_stage2.zip'